# Negation Augmentation Test

**Goal:** Test negation patterns on sample clauses from the LexGLUE unfair_tos dataset to verify the augmentation logic before full implementation.

## 1. Setup & Installation

In [1]:
# Install required libraries
!pip install -q datasets

In [2]:
# Import libraries
from datasets import load_dataset
import re

print("✓ Libraries imported successfully!")

✓ Libraries imported successfully!


## 2. Define Negation Function

In [3]:
def negate_clause(clause):
    """
    Apply negation to a risky clause.
    Returns (negated_clause, pattern_used) or (None, None) if negation doesn't make sense.
    """
    clause_lower = clause.lower()

    # Pattern 1: Modal verb "may"
    if ' may ' in clause_lower:
        negated = re.sub(r'\bmay\b', 'may not', clause, count=1, flags=re.IGNORECASE)
        return negated, "Modal: may → may not"

    # Pattern 2: Modal verb "can"
    elif ' can ' in clause_lower:
        negated = re.sub(r'\bcan\b', 'cannot', clause, count=1, flags=re.IGNORECASE)
        return negated, "Modal: can → cannot"

    # Pattern 3: Modal verb "will"
    elif ' will ' in clause_lower:
        negated = re.sub(r'\bwill\b', 'will not', clause, count=1, flags=re.IGNORECASE)
        return negated, "Modal: will → will not"

    # Pattern 4: Modal verb "shall"
    elif ' shall ' in clause_lower:
        negated = re.sub(r'\bshall\b', 'shall not', clause, count=1, flags=re.IGNORECASE)
        return negated, "Modal: shall → shall not"

    # Pattern 5: "reserve the right"
    elif 'reserve the right' in clause_lower:
        negated = re.sub(r'reserve the right', 'do not reserve the right', clause, count=1, flags=re.IGNORECASE)
        return negated, "Phrase: reserve the right → do not reserve the right"

    # Pattern 6: "has the right"
    elif 'has the right' in clause_lower:
        negated = re.sub(r'has the right', 'does not have the right', clause, count=1, flags=re.IGNORECASE)
        return negated, "Phrase: has the right → does not have the right"

    # Pattern 7: "have the right"
    elif 'have the right' in clause_lower:
        negated = re.sub(r'have the right', 'do not have the right', clause, count=1, flags=re.IGNORECASE)
        return negated, "Phrase: have the right → do not have the right"

    # Pattern 8: "is/are entitled to"
    elif 'is entitled to' in clause_lower or 'are entitled to' in clause_lower:
        negated = re.sub(r'is entitled to', 'is not entitled to', clause, count=1, flags=re.IGNORECASE)
        negated = re.sub(r'are entitled to', 'are not entitled to', negated, count=1, flags=re.IGNORECASE)
        return negated, "Phrase: entitled to → not entitled to"

    # Pattern 9: "agree to" / "agrees to"
    elif 'agree to' in clause_lower or 'agrees to' in clause_lower:
        negated = re.sub(r'agree to', 'do not agree to', clause, count=1, flags=re.IGNORECASE)
        negated = re.sub(r'agrees to', 'does not agree to', negated, count=1, flags=re.IGNORECASE)
        return negated, "Phrase: agree to → do not agree to"

    return None, None

print("✓ Negation function defined with 9 patterns")

✓ Negation function defined with 9 patterns


## 3. Load Dataset

In [4]:
print("Loading LexGLUE unfair_tos dataset...")
dataset = load_dataset("lex_glue", "unfair_tos")
train_data = dataset['train']

# Get label names
label_names = train_data.features['labels'].feature.names

print(f"\n✓ Dataset loaded successfully!")
print(f"  - Training examples: {len(train_data)}")
print(f"  - Risk categories: {len(label_names)}")
print(f"\nRisk Categories:")
for i, label in enumerate(label_names):
    print(f"  {i+1}. {label}")

Loading LexGLUE unfair_tos dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

unfair_tos/train-00000-of-00001.parquet:   0%|          | 0.00/501k [00:00<?, ?B/s]

unfair_tos/test-00000-of-00001.parquet:   0%|          | 0.00/147k [00:00<?, ?B/s]

unfair_tos/validation-00000-of-00001.par(…):   0%|          | 0.00/218k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5532 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1607 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2275 [00:00<?, ? examples/s]


✓ Dataset loaded successfully!
  - Training examples: 5532
  - Risk categories: 8

Risk Categories:
  1. Limitation of liability
  2. Unilateral termination
  3. Unilateral change
  4. Content removal
  5. Contract by using
  6. Choice of law
  7. Jurisdiction
  8. Arbitration


## 4. Test Negation on Sample Clauses

In [5]:
# Number of samples to test
NUM_SAMPLES = 20

print("=" * 100)
print(f"TESTING NEGATION ON {NUM_SAMPLES} RISKY CLAUSES")
print("=" * 100)

successful_negations = 0
failed_negations = 0

# Test on risky clauses
for i, example in enumerate(train_data):
    if successful_negations >= NUM_SAMPLES:
        break

    # Only process clauses with at least one risk label
    if len(example['labels']) > 0:
        original_clause = example['text']
        original_labels = [label_names[idx] for idx in example['labels']]

        # Try to negate
        negated_clause, pattern = negate_clause(original_clause)

        if negated_clause:
            successful_negations += 1
            print(f"\n{'─' * 100}")
            print(f"EXAMPLE {successful_negations}")
            print(f"{'─' * 100}")
            print(f"Pattern Used: {pattern}")
            print(f"\nOriginal Risks: {', '.join(original_labels)}")
            print(f"\nORIGINAL CLAUSE:")
            print(f"  {original_clause}")
            print(f"\nNEGATED CLAUSE (Safe):")
            print(f"  {negated_clause}")
        else:
            failed_negations += 1

TESTING NEGATION ON 20 RISKY CLAUSES

────────────────────────────────────────────────────────────────────────────────────────────────────
EXAMPLE 1
────────────────────────────────────────────────────────────────────────────────────────────────────
Pattern Used: Phrase: agree to → do not agree to

Original Risks: Contract by using

ORIGINAL CLAUSE:
  by creating a tinder account or by using the tinder imessage app ( `` tinder stacks '' ) , whether through a mobile device , mobile application or computer ( collectively , the `` service '' ) you agree to be bound by ( i ) these terms of use , ( ii ) our privacy policy and safety tips , each of which is incorporated by reference into this agreement , and ( iii ) any terms disclosed and agreed to by you if you purchase additional features , products or services we offer on the service ( collectively , this `` agreement '' ) . 


NEGATED CLAUSE (Safe):
  by creating a tinder account or by using the tinder imessage app ( `` tinder stacks ''

## 5. Summary Statistics

In [6]:
print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)
print(f"✓ Successful negations: {successful_negations}")
print(f"✗ Failed negations (no pattern matched): {failed_negations}")
print(f"Success rate: {successful_negations / (successful_negations + failed_negations) * 100:.1f}%")


SUMMARY
✓ Successful negations: 20
✗ Failed negations (no pattern matched): 5
Success rate: 80.0%


## 6. Pattern Coverage Analysis

In [7]:
print("\n" + "=" * 100)
print("PATTERN COVERAGE ANALYSIS")
print("=" * 100)

# Analyze which patterns are most common
pattern_counts = {}
total_risky = 0

for example in train_data:
    if len(example['labels']) > 0:
        total_risky += 1
        _, pattern = negate_clause(example['text'])
        if pattern:
            pattern_counts[pattern] = pattern_counts.get(pattern, 0) + 1

print(f"\nTotal risky clauses in dataset: {total_risky}")
print(f"Clauses that can be negated: {sum(pattern_counts.values())} ({sum(pattern_counts.values())/total_risky*100:.1f}%)")
print(f"\nPattern frequency:")
for pattern, count in sorted(pattern_counts.items(), key=lambda x: x[1], reverse=True):
    percentage = (count / total_risky) * 100
    print(f"  {pattern}: {count} clauses ({percentage:.1f}%)")


PATTERN COVERAGE ANALYSIS

Total risky clauses in dataset: 630
Clauses that can be negated: 444 (70.5%)

Pattern frequency:
  Modal: may → may not: 175 clauses (27.8%)
  Modal: will → will not: 110 clauses (17.5%)
  Modal: shall → shall not: 73 clauses (11.6%)
  Phrase: agree to → do not agree to: 45 clauses (7.1%)
  Phrase: reserve the right → do not reserve the right: 21 clauses (3.3%)
  Modal: can → cannot: 13 clauses (2.1%)
  Phrase: have the right → do not have the right: 5 clauses (0.8%)
  Phrase: has the right → does not have the right: 1 clauses (0.2%)
  Phrase: entitled to → not entitled to: 1 clauses (0.2%)
